In [1]:
import os
import shutil
import random

# --- Configuration ---
source_img_dir = "Annotated_Images/train/images"
source_lbl_dir = "Annotated_Images/train/labels" 
base_out_dir = "YOLO_Dataset"

# Splitting ratios
train_ratio, val_ratio, test_ratio = 0.7, 0.2, 0.1

# Create new directory structure
dirs_to_make = [
    f"{base_out_dir}/images/train", f"{base_out_dir}/labels/train",
    f"{base_out_dir}/images/val", f"{base_out_dir}/labels/val",
    f"{base_out_dir}/images/test", f"{base_out_dir}/labels/test"
]
for d in dirs_to_make:
    os.makedirs(d, exist_ok=True)

# Get all images and labels
images = [f for f in os.listdir(source_img_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
data_pairs = []

for img_name in images:
    lbl_name = os.path.splitext(img_name)[0] + ".txt"
    if os.path.exists(os.path.join(source_lbl_dir, lbl_name)):
        data_pairs.append((img_name, lbl_name))

print(f"Found {len(data_pairs)} matching image-label pairs.")

# Shuffle and split
random.seed(42)
random.shuffle(data_pairs)

total = len(data_pairs)
train_end = int(total * train_ratio)
val_end = train_end + int(total * val_ratio)

train_pairs = data_pairs[:train_end]
val_pairs = data_pairs[train_end:val_end]
test_pairs = data_pairs[val_end:]

def copy_files(pairs, split_name):
    for img_name, lbl_name in pairs:
        shutil.copy(os.path.join(source_img_dir, img_name), os.path.join(base_out_dir, f"images/{split_name}", img_name))
        shutil.copy(os.path.join(source_lbl_dir, lbl_name), os.path.join(base_out_dir, f"labels/{split_name}", lbl_name))
    print(f"Copied {len(pairs)} files to {split_name}.")

copy_files(train_pairs, "train")
copy_files(val_pairs, "val")
copy_files(test_pairs, "test")

Found 452 matching image-label pairs.
Copied 316 files to train.
Copied 90 files to val.
Copied 46 files to test.


In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import albumentations as A
import cv2
import os

# --- Configuration ---
train_img_dir = "YOLO_Dataset/images/train"
train_lbl_dir = "YOLO_Dataset/labels/train"
augment_multiplier = 2 # Creates 2 augmented copies per original image

# Define Albumentations Pipeline
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.MotionBlur(p=0.3),
    A.RandomScale(scale_limit=0.2, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=45, p=0.5)
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.2))

images = [f for f in os.listdir(train_img_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
total_augmented = 0

for img_name in images:
    img_path = os.path.join(train_img_dir, img_name)
    lbl_path = os.path.join(train_lbl_dir, os.path.splitext(img_name)[0] + ".txt")
    
    # Read image
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Read labels
    bboxes = []
    class_labels = []
    with open(lbl_path, 'r') as f:
        for line in f.readlines():
            parts = line.strip().split()
            if len(parts) == 5:
                class_labels.append(int(parts[0]))
                bboxes.append([float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])])
                
    # Generate augmented copies
    for i in range(augment_multiplier):
        try:
            augmented = transform(image=image, bboxes=bboxes, class_labels=class_labels)
            aug_img = augmented['image']
            aug_bboxes = augmented['bboxes']
            aug_labels = augmented['class_labels']
            
            # Save augmented image
            aug_img_name = f"aug_{i}_{img_name}"
            aug_img_path = os.path.join(train_img_dir, aug_img_name)
            cv2.imwrite(aug_img_path, cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))
            
            # Save augmented labels
            aug_lbl_name = os.path.splitext(aug_img_name)[0] + ".txt"
            aug_lbl_path = os.path.join(train_lbl_dir, aug_lbl_name)
            with open(aug_lbl_path, 'w') as f:
                for bbox, label in zip(aug_bboxes, aug_labels):
                    f.write(f"{label} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n")
            
            total_augmented += 1
        except Exception as e:
            # ShiftScaleRotate can sometimes push bboxes out of bounds entirely
            pass

print(f"Augmentation complete. Generated {total_augmented} new training images.")

c:\Users\bombo\anaconda3\Lib\site-packages\albumentations\__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.5'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
c:\Users\bombo\anaconda3\Lib\site-packages\albumentations\core\validation.py:87: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Augmentation complete. Generated 632 new training images.


In [3]:
import yaml
from ultralytics import YOLO
import os

# 1. Create the dataset.yaml file required by YOLO
yaml_content = {
    'path': os.path.abspath('YOLO_Dataset'),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {
        0: '1-ball',
        1: '2-ball',
        2: '3-ball',
        3: '4-ball',
        4: '5-ball',
        5: '6-ball',
        6: '7-ball',
        7: '8-ball',
        8: '9-ball',
        9: 'cue',
        10: 'rack'
    }
}

with open('dataset.yaml', 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print("dataset.yaml created successfully.")

# 2. Download YOLOv11n and Train
# YOLOv11 was recently released. Ultralytics automatically downloads it when called.
model = YOLO('yolo11n.pt') 

print("Starting training...")
results = model.train(
    data='dataset.yaml',
    epochs=100,
    imgsz=416,
    batch=8,
    patience=20,     # Early stopping
    workers=0,
    project='Pool_Detection',
    name='yolo11n_run'
)

dataset.yaml created successfully.
Starting training...
New https://pypi.org/project/ultralytics/8.4.38 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.115  Python-3.12.7 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: task=detect, mode=train, model=yolo11n.pt, data=dataset.yaml, epochs=100, time=None, patience=20, batch=8, imgsz=416, save=True, save_period=-1, cache=False, device=None, workers=0, project=Pool_Detection, name=yolo11n_run, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=N

train: Scanning C:\Users\Axioo Pongo\Desktop\All_in_One\College\Tugas Akhir\Pre-Processing\4_Second_Training\YOLO_Dataset\labels\train... 948 images, 12 backgrounds, 0 corrupt: 100%|██████████| 948/948 [00:01<00:00, 620.45it/s] 

train: New cache created: C:\Users\Axioo Pongo\Desktop\All_in_One\College\Tugas Akhir\Pre-Processing\4_Second_Training\YOLO_Dataset\labels\train.cache


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access  (ping: 0.10.0 ms, read: 41.716.0 MB/s, size: 291.2 KB)


val: Scanning C:\Users\Axioo Pongo\Desktop\All_in_One\College\Tugas Akhir\Pre-Processing\4_Second_Training\YOLO_Dataset\labels\val... 90 images, 4 backgrounds, 0 corrupt: 100%|██████████| 90/90 [00:00<00:00, 587.12it/s]

...

    100/100     0.582G     0.9342     0.4823     0.7911          7        416: 100%|██████████| 119/119 [00:45<00:00,  2.61it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:01<00:00,  3.26it/s]

                   all         90        504      0.893      0.897      0.918      0.631



100 epochs completed in 0.876 hours.
Optimizer stripped from Pool_Detection\yolo11n_run\weights\last.pt, 5.4MB
Optimizer stripped from Pool_Detection\yolo11n_run\weights\best.pt, 5.4MB

Validating Pool_Detection\yolo11n_run\weights\best.pt...
Ultralytics 8.3.115  Python-3.12.7 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11n summary (fused): 100 layers, 2,584,297 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:02<00:00,  2.89it/s]


                   all         90        504      0.898      0.888      0.963      0.681
                1-ball          6          6      0.832      0.828      0.924      0.642
                2-ball         22         22          1      0.851      0.983      0.657
                3-ball         39         40      0.951      0.925      0.959      0.646
                4-ball         44         44      0.949      0.845      0.967      0.653
                5-ball         50         50      0.997       0.94      0.988      0.685
                6-ball         62         62          1      0.787      0.937      0.683
                7-ball         61         61      0.988      0.869      0.971      0.688
                8-ball         66         66      0.967      0.882      0.916      0.675
                9-ball         73         73      0.939      0.904      0.974        0.7
                   cue         79         79          1      0.936      0.982      0.665
                  rac

In [5]:
from ultralytics import YOLO

# Load the best weights from your training run
# (Ensure this path matches the output path from Cell 3)
best_model_path = 'Pool_Detection/yolo11n_run/weights/best.pt'
model = YOLO(best_model_path)

print("Evaluating model on the Test Split...")

# Validate against the 'test' split defined in dataset.yaml
metrics = model.val(data='dataset.yaml', split='test')

# Print core metrics
print(f"Mean Average Precision (mAP@50): {metrics.box.map50:.3f}")
print(f"Mean Average Precision (mAP@50-95): {metrics.box.map:.3f}")

# Optional: Run inference on a random test image to visually inspect
# import glob
# test_images = glob.glob('YOLO_Dataset/images/test/*.jpg')
# if test_images:
#    model.predict(source=test_images[0], save=True, imgsz=416)

Evaluating model on the Test Split...
Ultralytics 8.3.115  Python-3.12.7 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11n summary (fused): 100 layers, 2,584,297 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1276.1358.2 MB/s, size: 333.4 KB)


val: Scanning C:\Users\Axioo Pongo\Desktop\All_in_One\College\Tugas Akhir\Pre-Processing\4_Second_Training\YOLO_Dataset\labels\test.cache... 46 images, 0 backgrounds, 0 corrupt: 100%|██████████| 46/46 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:06<00:00,  2.08s/it]


                   all         46        275      0.908      0.849      0.906      0.629
                1-ball          7          7      0.822      0.665      0.632      0.421
                2-ball         18         18      0.984      0.722      0.875      0.613
                3-ball         22         23      0.904      0.818      0.834      0.599
                4-ball         24         24      0.883      0.958      0.937      0.675
                5-ball         29         29          1      0.911      0.964      0.671
                6-ball         27         27      0.954      0.769      0.899      0.629
                7-ball         26         26      0.971      0.846      0.957      0.613
                8-ball         37         37      0.976      0.865      0.943      0.632
                9-ball         39         39      0.848      0.897      0.958       0.69
                   cue         44         44          1      0.882       0.97      0.686
                  rac